In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# ============================
# 1. Load Weather + PM dataset
# ============================
weather_pm = pd.read_excel("/Users/mac/Desktop/DataVisualization/New_PM/weather_pm_merged_final.xlsx")

weather_pm["Date"] = pd.to_datetime(weather_pm["Date"])
weather_pm["Year"] = weather_pm["Date"].dt.year

# ============================
# 2. Load GHG dataset
# ============================
ghg = pd.read_excel("/Users/mac/Desktop/DataVisualization/New_PM/GHGFOR PALESTINE.xlsx", skiprows=3)

ghg.columns = ["Year", "CO2", "CH4", "N2O", "Total_GHG"]

ghg = ghg.dropna(subset=["Year"])
ghg["Year"] = ghg["Year"].astype(int)

for col in ["CO2", "CH4", "N2O", "Total_GHG"]:
    ghg[col] = pd.to_numeric(ghg[col], errors="coerce")

ghg["GHG_Source"] = "Observed_PCBS"

# ============================
# 3. Estimate GHG for 2024–2025
# ============================
future_years = pd.DataFrame({"Year": [2024, 2025]})

estimated = future_years.copy()

for col in ["CO2", "CH4", "N2O", "Total_GHG"]:
    train = ghg.dropna(subset=[col]).tail(5)

    X = train[["Year"]]
    y = train[col]

    model = LinearRegression()
    model.fit(X, y)

    estimated[col] = model.predict(future_years[["Year"]])

estimated["GHG_Source"] = "Estimated_linear_trend"

# Combine observed + estimated GHG
ghg_full = pd.concat([ghg, estimated], ignore_index=True)

# ============================
# 4. Merge Weather + PM + GHG
# ============================
final_dataset = weather_pm.merge(
    ghg_full,
    on="Year",
    how="left"
)

# ============================
# 5. Remove unnecessary column
# ============================
final_dataset = final_dataset.drop(columns=["GHG_Source"])

# ============================
# 6. Save final dataset
# ============================
final_dataset.to_excel("weather_pm_ghg_final.xlsx", index=False)
final_dataset.to_csv("weather_pm_ghg_final.csv", index=False)

# ============================
# 7. Check result
# ============================
print(final_dataset.info())
print(final_dataset.isna().sum())
print(final_dataset.head())

<class 'pandas.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Station      366 non-null    str           
 1   Date         366 non-null    datetime64[us]
 2   Temperature  366 non-null    float64       
 3   Sunshine     366 non-null    float64       
 4   RH           366 non-null    float64       
 5   PRESS        366 non-null    float64       
 6   Rainfall     366 non-null    float64       
 7   Wind         366 non-null    float64       
 8   Month        366 non-null    int64         
 9   Year         366 non-null    int32         
 10  Day          366 non-null    int64         
 11  DayOfWeek    366 non-null    int64         
 12  AQI          366 non-null    int64         
 13  PM2.5        366 non-null    float64       
 14  CO2          366 non-null    float64       
 15  CH4          366 non-null    float64       
 16  N2O          366 no

In [4]:
import pandas as pd

# Load your merged dataset
df = pd.read_excel("weather_pm_ghg_final.xlsx")

# Remove AQI column
df = df.drop(columns=["AQI"])

# Save the updated dataset
df.to_excel("weather_pm_ghg_final_noAQI.xlsx", index=False)
df.to_csv("weather_pm_ghg_final_noAQI.csv", index=False)

# Verify
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Station      366 non-null    str           
 1   Date         366 non-null    datetime64[us]
 2   Temperature  366 non-null    float64       
 3   Sunshine     366 non-null    float64       
 4   RH           366 non-null    float64       
 5   PRESS        366 non-null    float64       
 6   Rainfall     366 non-null    float64       
 7   Wind         366 non-null    float64       
 8   Month        366 non-null    int64         
 9   Year         366 non-null    int64         
 10  Day          366 non-null    int64         
 11  DayOfWeek    366 non-null    int64         
 12  PM2.5        366 non-null    float64       
 13  CO2          366 non-null    float64       
 14  CH4          366 non-null    float64       
 15  N2O          366 non-null    float64       
 16  Total_GHG    366 no

In [5]:
# Remove the Total_GHG column
final_dataset = final_dataset.drop(columns=["Total_GHG"])

# Save to Excel and CSV
final_dataset.to_excel("weather_pm_ghg_final_clean.xlsx", index=False)
final_dataset.to_csv("weather_pm_ghg_final_clean.csv", index=False)

print("Total_GHG removed successfully and cleaned dataset saved.")
print(final_dataset.columns)

Total_GHG removed successfully and cleaned dataset saved.
Index(['Station', 'Date', 'Temperature', 'Sunshine', 'RH', 'PRESS', 'Rainfall',
       'Wind', 'Month', 'Year', 'Day', 'DayOfWeek', 'AQI', 'PM2.5', 'CO2',
       'CH4', 'N2O'],
      dtype='str')


In [6]:
# Remove AQI from the dataset
final_dataset = final_dataset.drop(columns=["AQI"])

# Save the updated dataset
final_dataset.to_excel("weather_pm_ghg_final_clean.xlsx", index=False)
final_dataset.to_csv("weather_pm_ghg_final_clean.csv", index=False)

print(final_dataset.columns)

Index(['Station', 'Date', 'Temperature', 'Sunshine', 'RH', 'PRESS', 'Rainfall',
       'Wind', 'Month', 'Year', 'Day', 'DayOfWeek', 'PM2.5', 'CO2', 'CH4',
       'N2O'],
      dtype='str')
